# Setup

In [ ]:
# Auto-install anything missing, as 02b does. gt/gtExtras/gtsummary are the usual
# culprits on a fresh cloud environment. Note lme4 Depends on Matrix, so attaching it
# here also makes S4 subsetting of the kinship matrix work.
pkgs <- c('data.table','ggplot2','ggrepel','gt','gtExtras','gtsummary',
          'lme4','lme4qtl','lmerTest','lmtest','parallel','RhpcBLASctl')
if(!require(lme4qtl)) { remotes::install_github("variani/lme4qtl") } # Special install
lapply(pkgs, \(pkg) { if(!require(pkg, character.only=T)) {install.packages(pkg);require(pkg,character.only=T)}}) |> invisible()

options(datatable.na.strings=c('NA',''))

# As in 02b: without this each forked worker spawns its own BLAS/OpenMP threads, they
# oversubscribe the cores, and a fit that should take seconds takes a minute (measured
# 2026-09-20: 73 s elapsed for 494 s of CPU).
blas_set_num_threads(1)
omp_set_num_threads(1)

# Every input this notebook does not produce itself is pulled from the workspace
# bucket, so a fresh cloud environment can run it without depending on files an
# earlier session happened to leave on the persistent disk.
ws_bucket <- Sys.getenv('WORKSPACE_BUCKET',
                        unset='gs://fc-secure-4a392455-5587-4d6f-b8bd-01a1f834ae63')
grab_file_if_not_extant <- \(f) if(!file.exists(f)) {
  dir.create(dirname(f), recursive=TRUE, showWarnings=FALSE)
  system(paste0('gcloud storage cp ', ws_bucket, '/', f, ' ', f)) # Needs a local destination, not just the source.
}

grab_file_if_not_extant('data/derived/analysis_df-fhs.csv')                       # 03b
grab_file_if_not_extant('data/derived/genomics/fhs_km.RData')                     # 03b
grab_file_if_not_extant('data/derived/metabolomics/QCd/merged_QCd-aligned.csv')   # 03a
grab_file_if_not_extant('data/derived/metabolomics/met_info-aligned.csv')         # 03a
grab_file_if_not_extant('data/derived/metabolomics/met_info-mesa.csv')            # 01a

# --- Lanes ---------------------------------------------------------------------
# A lane is one declared (G, Y, E) triple: a SNP, the outcome, and the exposure whose
# interaction with it is followed up. anchors.csv declares them, and every consumer
# (01c, 02a, 02b, 02c, 03c, manuscript.Rmd) reads that one file, so adding or parking a
# lane is one row there rather than an edit in each notebook. On Terra only *.ipynb
# reaches the VM, so the copy read is the one published to the workspace bucket.

# The trait a variable measures, ignoring how it was derived: bmi, bmi_covariate and
# bmi_wins are all BMI. Extends the suffix convention 01c's observed() uses.
trait <- \(v) sub('_(log|wins|trunc|covariate)$', '', v)

read_lanes <- function(bucket) {
  lanes <- fread(cmd = paste0('gcloud storage cat ', bucket, '/anchors.csv'))
  miss  <- setdiff(c('lane_id','G','Y','E','rsid','gene','label','include'), names(lanes))
  if (length(miss))
    stop('anchors.csv is missing column(s) ', paste(miss, collapse=', '),
         ' -- has the current version been published to the bucket? See the README.',
         call.=FALSE)
  if (!'extra_covars' %in% names(lanes)) lanes[, extra_covars := '']
  lanes[, extra_covars := fifelse(is.na(extra_covars), '', as.character(extra_covars))]
  lanes[, include := as.logical(include)]
  if (anyDuplicated(lanes$lane_id))
    stop('anchors.csv: lane_id must be unique; duplicated: ',
         paste(unique(lanes$lane_id[duplicated(lanes$lane_id)]), collapse=', '), call.=FALSE)
  # Unique by trait, not just by name, so ensure_lane_id() below can match on traits
  # without ambiguity.
  if (anyDuplicated(lanes[, .(G, trait(Y), trait(E))]))
    stop('anchors.csv: two lanes declare the same SNP, outcome trait and exposure trait.', call.=FALSE)
  # A lane with one trait on both sides would fit e.g. bmi ~ G*bmi_covariate. Compared by
  # trait rather than by name, so that case is caught and not only bmi ~ G*bmi.
  same <- lanes[trait(Y) == trait(E), lane_id]
  if (length(same))
    stop('anchors.csv: outcome and exposure are the same trait in ',
         paste(same, collapse=', '), '. A lane cannot use one trait on both sides.', call.=FALSE)
  lanes[]
}

# Covariates for one model: the shared set plus a lane's own additions, minus anything
# measuring that model's outcome or exposure. By trait, because a name-level setdiff
# misses exactly the case that matters -- bmi_covariate as a precision covariate in a
# lane whose outcome is bmi survives setdiff(covars, c('bmi','mvpa_wins')).
lane_covars <- function(covars, Y, E = NULL, extra = '') {
  extra <- if (length(extra) == 1 && !is.na(extra) && nzchar(extra))
             strsplit(trimws(extra), '[ ;,]+')[[1]] else character()
  cs <- unique(c(covars, extra))
  cs[!trait(cs) %in% trait(c(Y, E))]
}

# Results checkpointed before lanes existed carry (G, Y, E) but no lane_id. Attach it
# rather than refusing them, so a notebook still reads older results without re-fitting.
# Matched on the SNP and the outcome and exposure TRAITS, so a result written when the
# CETP exposure was still named bmi attaches to the lane that now names it bmi_covariate.
# Rows matching no declared lane keep lane_id = NA.
ensure_lane_id <- function(dt, lanes) {
  if ('lane_id' %in% names(dt) || !all(c('G','Y','E') %in% names(dt))) return(dt)
  key <- lanes[, .(lane_id, G, Y_trait = trait(Y), E_trait = trait(E))]
  key[copy(dt)[, `:=`(Y_trait = trait(Y), E_trait = trait(E))], on=.(G, Y_trait, E_trait)
    ][, c('Y_trait', 'E_trait') := NULL][]
}

# Workaround so that lmerTest works on lme4qtl model objects, so we can get lmerTest Satterthwaite p-values.
myLmer <- \(formula, data, relmat) {
  # method.relfac='chol' skips relfac's duplicated-column scan, which extracts every
  # column of the kinship matrix as a dense vector on EVERY fit -- far more expensive
  # than the factorization itself, and it can divert to a dense eigendecomposition.
  # Results are bit-identical.
  model <- relmatLmer(formula, data, relmat=relmat, method.relfac='chol')
  lmerTest:::as_lmerModLT(model, as.function(model))
}

In [ ]:
data <- fread('data/derived/analysis_df-fhs.csv')
variants_of_interest <- fread(cmd=paste0('gcloud storage cat ', ws_bucket, '/variants_of_interest.csv'))
cpaids <- variants_of_interest[cpaid %in% names(data), cpaid]
lanes <- read_lanes(ws_bucket)

data <- fread('data/derived/analysis_df-fhs.csv')[
  ][, metabolomics_visit := as.factor(metabolomics_visit)
  # Variants are numeric (0/1/2), but we temporarily set to character so they don't get scale()'d.
  ][, names(.SD) := lapply(.SD, as.character), .SDcols=cpaids
  ][, names(.SD) := lapply(.SD, scale       ), .SDcols=is.numeric
  ][, names(.SD) := lapply(.SD, as.numeric  ), .SDcols=cpaids
]

load('data/derived/genomics/fhs_km.RData') # Kinship (ids remapped in 03b)
# NB this matrix was sparsified in 03b: relatedness below 0.025 zeroed, 0.01 ridge on the
# diagonal. The raw matrix is 32% non-zero and makes every fit take hours. See the decision
# record in 03b for the sweep, the statistical implications, and what the manuscript needs
# to state. If fits are unexpectedly slow, check that this file is the sparsified one:
#   Matrix::nnzero(km) / nrow(km)^2   should be ~0.0013, not ~0.32
data <- data[NWD_Id %in% colnames(km)]
unsuitable_snps <- data[, names(.SD)[sapply(.SD, \(x) all(is.na(x)) | sum(x,na.rm=T)<10)], .SDcols=patterns('^chr')]
data <- data[, .SD, .SDcols=!unsuitable_snps]
cpaids <- setdiff(cpaids,unsuitable_snps)

data[ # dosages at the lane SNPs
  ][, .SD, .SDcols=intersect(unique(lanes$G), cpaids)
  ] |>
  tbl_summary(
    missing='no',
    statistic = list(
      all_continuous()  ~ c("{mean} ± {sd}"),
      all_categorical() ~ c("{n} ({p}%)"   )
  )) |> add_n() |> as_gt() |> gt:::as.tags.gt_tbl()

met_nms <- names(fread('data/derived/metabolomics/QCd/merged_QCd-aligned.csv',nrows=0,drop=1))
covars <- c('sex', 'age', 'smoking', 'metabolomics_visit', paste0('gPC',1:9)) # 9 gPCs, as in 02b.
stopifnot('missing gPCs -- check 03b merged freeze9_pcair_results' =
          all(paste0('gPC',1:9) %in% names(data)))

# FHS metabolomics here is cross-sectional -- one sample per person -- so the models carry
# only the kinship-structured intercept. 02b additionally has an iid permanent-individual
# term for MESA's repeated exams; that would be confounded with the residual here. FHS
# relatedness is denser than MESA's, so this single term does correspondingly more work.
stopifnot('FHS frame is not one row per person -- revisit the RE structure' =
          data[, .N, by=NWD_Id][, max(N)] == 1)
# The confirmation models below are about the declared lanes, not the whole screened
# panel: FHS is asked one question -- do each lane's interaction and its component
# associations hold here too. Lanes come from anchors.csv, the same declaration 02b
# screens, so the two notebooks cannot disagree about which loci they are about.
anchor_snps <- intersect(unique(lanes$G), cpaids)   # Lane SNPs usable in this frame.
stopifnot('no lane SNP survives in the FHS frame' = length(anchor_snps) > 0)
cat('Lane SNPs in FHS:', paste(anchor_snps, collapse=', '), '\n')

calc_eff_n_metabolites <- \(met_nms) {
  met_nms <- met_nms[!is.na(met_nms)] |> unique()
  met_mtx <- data[ rowSums(is.na(data[,..met_nms]))==0, ..met_nms ] |> as.matrix(rownames=1) # Only samples having data for ALL metabolomics methods
  met_eigvals <- prcomp(met_mtx, scale=T, center=T)$sdev^2
  met_eff_n <- sum(met_eigvals)^2 / sum(met_eigvals^2)
}

# Map aligned vs. MESA-specific metabolites
This code was used to map MESA -> FHS metabolites, so that we only rerun the metabolites that were already significant in MESA. But now, we want to run FHS from scratch regardless of what we found in MESA. Hence why this piece is now commented out.

Not all can be mapped, but we try our best.

In [ ]:
#met_info_mesa  <- fread('data/derived/metabolomics/met_info-mesa.csv'   )
#met_info_align <- fread('data/derived/metabolomics/met_info-aligned.csv')
#
## Fixing differences in the formats I made for these two files:
## * "_pos"/"_neg" at the end of the unique_met_ids
## * "Amide-neg" vs. "Amide-Negative-sMRM" Method name
#met_info_align[Method=='Amide-Negative-sMRM', Method := 'Amide-neg']
#met_info_mesa [, unique_met_id := sub('_pos','',sub('_neg','',unique_met_id))]
#
#signif_in_mesa <- fread('runs.csv')[grepl('chr14',snp) & !single_exam & (signif_funnel | signif_mwis), met]
#signif_in_mesa <- sub('_pos','',sub('_neg','',signif_in_mesa)) # "_pos"/"_neg" at the end of ids fix (messy)
#signif_in_mesa <- met_info_mesa[unique_met_id %in% signif_in_mesa]
#
#met_info_align$unique_met_id %in% signif_in_mesa$unique_met_id |> sum() # Aw that's only like half :(
#tmp1 <- met_info_align[Method!='Amide-neg'][signif_in_mesa[Method!='Amide-neg'], on=.(unique_met_id)]
#tmp2 <- met_info_align[Method=='Amide-neg'][signif_in_mesa[Method=='Amide-neg'], on=.(  Metabolite )][, unique_met_id := i.unique_met_id] # I manually generated the unique ids for the amines arbitrarily. Must match on name instead.
#tmp <- rbind(tmp1,tmp2,fill=T)
#
## Sanity check that the QI ids do correspond between datasets, its just that not all the MESA metabolites are in the aligned dataset.
#tmp[, all(      Compound_ID==      i.Compound_ID | is.na(      Compound_ID))]
#tmp[, all(Pilot_Compound_ID==i.Pilot_Compound_ID | is.na(Pilot_Compound_ID))]
#
#met_nms <- tmp[Method=='Amide-neg' | !is.na(Compound_ID), unique_met_id]
#
#met_nms <- met_nms[met_nms %in% names(data)] # To account for metabolites lost during QC for high missingness.

# Confirm known associations, choose SNPs to pursue
## Define formulas

In [ ]:
# One row per model, keyed off the lanes. A model two lanes share (both CETP lanes fit
# bmi_covariate ~ CETP) is fit once and labelled with both. Exposures FHS does not carry -- it has no
# physical activity measure -- rule out that lane's E~G and Y~G*E, but not its Y~G.
lanes_here <- lanes[G %in% anchor_snps & Y %in% names(data)]
E_ok <- \(E) E %in% names(data) & E != 'smoking'   # Categorical smoking adds complication; not worth it for one SNP.

runs1 <- rbindlist(fill=TRUE, list(
  lanes_here[E_ok(E), .(lane_id=paste(lane_id, collapse=';')), by=.(Y, E)][, model := 'Y~E'],
  lanes_here[,        .(lane_id=paste(lane_id, collapse=';')), by=.(Y, G)][, model := 'Y~G'],
  lanes_here[E_ok(E), .(lane_id=paste(lane_id, collapse=';')), by=.(E, G)][, model := 'E~G'],
  lanes_here[E_ok(E), .(lane_id, Y, E, G, extra_covars)][,                    model := 'Y~G*E']))[
  ][, `:=`(M=NA, exam='all', lm_or_lmm='LMM')
  ][is.na(extra_covars), extra_covars := ''

  ][model=='Y~E',   fmla := paste0('<Y> ~ <E>')
  ][model=='E~G',   fmla := paste0('<E> ~ <G>')
  ][model=='Y~G',   fmla := paste0('<Y> ~ <G>')
  # G:sex is a standard term in 02b's interaction models, so Y~G*E carries it here too.
  # (02b omits it from the marginal Y~G and E~G tables, so it is attached per model.)
  ][model=='Y~G*E', fmla := paste0('<Y> ~ <G>*<E> + <G>*sex')

  # Covariates minus each model's own traits; a lane's additions enter its interaction only.
  ][, fmla := paste(fmla, '+', mapply(Y, E, fifelse(model=='Y~G*E', extra_covars, ''), USE.NAMES=FALSE,
        FUN=\(Y, E, x) paste(lane_covars(covars, Y, E, x), collapse='+')))
  ][model=='Y~G*E', fmla := paste(fmla, '+',                 # Add gPC*E term for models w/ G*E focal term.
     paste(collapse='+', paste0('gPC',1:9,'*<E>')))
  ][, fmla := mapply(fmla, Y, G, E, USE.NAMES=FALSE, FUN=\(f, Y, G, E) { # Replace placeholders.
        if (!is.na(Y)) f <- gsub('<Y>', Y, f)
        if (!is.na(G)) f <- gsub('<G>', G, f)
        if (!is.na(E)) f <- gsub('<E>', E, f)
        f })
  ][, fmla := paste(fmla, '+', '(1|NWD_Id)')                 # Add random intercept.

  # Specify the term of interest whose stats will be extracted by specifying a pattern to grep.
  #   Interaction terms may be named either "X:Y" or "Y:X", need to account for both....
  ][model=='Y~E',   term_pat := paste0('^',E,'$')
  ][model=='E~G',   term_pat := paste0('^',G,'$')
  ][model=='Y~G',   term_pat := paste0('^',G,'$')
  ][model=='Y~G*E', term_pat := paste0(G,':',E,'|',E,':',G)
]

## Confirm Y ~ E assocations
Model: `Y ~ E + covariates + (1|ID)`
Only top 10 model terms are shown. E is highlighted.

In [ ]:
tbls <- runs1[model=='Y~E', .(Map(fmla,Y,E, f=\(fmla,Y,E) {
  myLmer(fmla, data, list(NWD_Id=km)) |>
  summary() |> coef() |>
  as.data.table(keep.rownames='term') |>
  (\(dt) dt[order(`Pr(>|t|)`)])() |>
  head(10) |> gt(caption=paste(Y,'~',E)) |>
  gt_highlight_rows(rows=term==E) |>        # Exact: grepl('bmi') would also light up bmi_covariate.
  gt:::as.tags.gt_tbl()
}))]

tbls$V1

## Confirm Y ~ G associations

In [ ]:
options(mc.cores=4L)
i <- 0
runs1[
  ][model=='Y~G' & exam=='all'
    , c('est','se','df','t','p','n') := transpose(mcMap(fmla,term_pat, f=\(fmla,pat) {
        message('  ',(i<<-i+1)*getOption('mc.cores'),'/',.N,'\r',appendLF=F) # Progress bar
        model <- myLmer(fmla,data,list(NWD_Id=km))
        coefs <- summary(model)$coefficients
        c(coefs[grepl(pat,rownames(coefs)),],
          nrow(model@frame)                 )
      }))
]

In [ ]:
# Display
variants_of_interest[
  ][runs1, on=.(cpaid=G)
  ][model=='Y~G' & exam=='all'
  ][order(p)
  ][, .(lane_id,Y,G=cpaid,E,Gene=gene,rsID=rsid,analysis,Annotation=annotation,G_p=signif(p,3), G_β=signif(est,3),`N (all timepoints)`=n)
] |> gt(caption='G main effects with p<0.05 are highlighted') |> gt_highlight_rows(rows=G_p<0.05) |> gt:::as.tags.gt_tbl()

## Confirm Y ~ GxE associations
Model: `Y ~ G*E + G*sex + covars + (1|ID)`

In [ ]:
options(mc.cores=4L)
i <- 0
runs1[
  ][model=='Y~G*E' & exam=='all'
    , c('est','se','df','t','p','n') := transpose(mcMap(fmla,term_pat, f=\(fmla,pat) {
        message('  ',(i<<-i+1)*getOption('mc.cores'),'/',.N,'\r',appendLF=F) # Progress bar
        model <- myLmer(fmla,data,list(NWD_Id=km))
        coefs <- summary(model)$coefficients
        c(coefs[grepl(pat,rownames(coefs)),],
          nrow(model@frame)                 )
      }))
]

In [ ]:
# Display
variants_of_interest[
  ][runs1, on=.(cpaid=G)
  ][model=='Y~G*E' & exam=='all'
  ][order(p)
  ][, .(lane_id,Y,G=cpaid,E,Gene=gene,rsID=rsid,analysis,Annotation=annotation,GxE_p=signif(p,3), GxE_β=signif(est,3),`N (all timepoints)`=n)
] |> gt(caption='GxEs with p<0.05 are highlighted') |> gt_highlight_rows(rows=GxE_p<0.05) |> gt:::as.tags.gt_tbl()

# Replicate the MESA GxM hits

02b writes `results/mesa_signif_GxMs.csv`: the metabolite x SNP interactions that passed
the metabolome-wide screen in MESA. This section tests those same interactions in FHS and
nothing else -- a handful of models rather than a second screen, which is what makes these
fits affordable given how dense the FHS kinship matrix is.

Two restrictions, both deliberate:

* **Primary GxM interaction only.** No `G:E` adjustment and no mediation analysis. The
  `G:E` term is a single commented line in the formula below if that changes.
* **Kinship-only random effect.** FHS metabolomics is cross-sectional, one sample per
  person, so there is no repeated-measures term to carry. 02b additionally fits an iid
  permanent-individual effect for MESA's repeated exams; here that would be confounded
  with the residual. Setup asserts the one-row-per-person assumption.

In [ ]:
grab_file_if_not_extant('results/mesa_signif_GxMs.csv')   # Written by 02b.
mesa_hits <- ensure_lane_id(fread('results/mesa_signif_GxMs.csv'), lanes) # Hits written before lanes carry no lane_id.
mesa_hits <- lanes[, .(lane_id, label, include, extra_covars)][
  mesa_hits[, !intersect(c('label','include','extra_covars'), names(mesa_hits)), with=FALSE], on='lane_id']
setnames(mesa_hits, '\u03b2_GxM', 'mesa_beta', skip_absent=TRUE)  # ASCII name for the beta column.
setnames(mesa_hits, 'p', 'mesa_p', skip_absent=TRUE)

# MESA ids carry an ionization suffix the aligned dataset does not:
#   QI14545_C18_neg -> QI14545_C18
# Amide-neg would NOT be mappable this way -- 01a builds those ids from a row index, so they
# do not correspond across datasets at all -- but no Amide-neg feature reached the hit list,
# so every hit here is addressable. The check below catches it if that ever changes.
mesa_hits[, M_aligned := sub('_(pos|neg)$', '', M)]

# Testability depends on the METABOLITE only. The GxM model below is Y ~ G*M + covars and
# carries no E term at all -- the exposure matters solely for the G:E adjustment, which is
# descoped. So a hit from an anchor whose exposure FHS lacks is still perfectly testable:
# what cannot be done there is adjusting for G:E, not the GxM test itself.
mesa_hits[, met_present      := M_aligned %in% met_nms]
mesa_hits[, exposure_present := E %in% names(data)]   # Only gates the optional G:E adjustment.
mesa_hits[, testable         := met_present]

cat('MESA hits:', nrow(mesa_hits), '| testable in FHS:', mesa_hits[, sum(testable)], '\n\n')
print(mesa_hits[, .(hits=.N, testable=sum(testable),
                    metabolite_absent=sum(!met_present),
                    GxE_adjustable=sum(exposure_present)),
                by=.(lane_id, rsid, gene, Y, E)])

if (mesa_hits[, any(!exposure_present)])
  # NB `mesa_hits[!exposure_present]` reads as data.table's not-join syntax, not as
  # logical negation, and fails looking the symbol up in the calling scope.
  cat('\nNOTE:', paste(mesa_hits[exposure_present == FALSE, unique(E)], collapse=', '),
      'is absent from the FHS frame. Those hits are still tested -- the GxM model needs\n',
      ' no exposure term -- but the G:E adjustment could never be added for them, and the\n',
      ' confirmation section above cannot fit their E~G or Y~G*E either. Physical activity\n',
      ' is searched for in 00_picsure but never coalesced into a named column, so it never\n',
      ' reaches analysis_df-fhs; adding it upstream would lift both restrictions.\n')

hits <- mesa_hits[testable == TRUE]
if (!nrow(mesa_hits))
  cat('\n02b reported no hit at its screen threshold, so there is nothing to replicate',
      'per feature.\n The metabolome-wide screen below is the section that runs in that case.\n')
if (nrow(mesa_hits) && !nrow(hits))
  warning('MESA reported ', nrow(mesa_hits), ' hit(s) but none is testable in FHS -- check the ',
          'id mapping and the exposures.', call.=FALSE)

In [ ]:
# Match 02b's MWIS specification as closely as FHS allows: same covariates, the same G:sex
# term, the same gPC x M interactions -- so an estimate here is comparable to the MESA one
# rather than merely similar. The one intentional omission is the G:E adjustment; uncomment
# the marked line to restore it.
if (nrow(hits)) {
hits[, fmla := paste0(Y, ' ~ ', G, '*', M_aligned)
  ][, fmla := paste(fmla, '+', mapply(Y, E, fifelse(is.na(extra_covars), '', extra_covars), USE.NAMES=FALSE,
        FUN=\(Y, E, x) paste(lane_covars(covars, Y, E, x), collapse='+'))) # Covars (+ lane additions), minus Y's and E's traits.
  ][, fmla := paste(fmla, '+', paste0(G, ':sex'))
  ][, fmla := paste(fmla, '+', paste(collapse='+', paste0('gPC', 1:9, '*', M_aligned)))
  # ][, fmla := paste(fmla, '+', paste0(G, '*', E))   # G:E adjustment -- descoped, see above.
  ][, fmla := paste(fmla, '+ (1|NWD_Id)')
  ][, term_pat := paste0(G, ':', M_aligned, '|', M_aligned, ':', G)
]

options(mc.cores = min(4L, nrow(hits)))
i <- 0
hits[, c('fhs_est','fhs_se','fhs_df','fhs_t','fhs_p','fhs_n') :=
  transpose(mcMap(fmla, term_pat, f = \(fmla, pat) {
    message('  ', (i <<- i+1)*getOption('mc.cores'), '/', .N, '\r', appendLF=FALSE)
    model <- myLmer(fmla, data, list(NWD_Id=km))
    coefs <- summary(model)$coefficients
    c(coefs[grepl(pat, rownames(coefs)), ], nrow(model@frame))
  }))]
} else cat('No testable hit: skipping the per-feature replication fits.\n')

## Replication summary

These are pre-specified hypotheses rather than a screen, so the yardstick is Bonferroni
over the hits actually tested, not the metabolome-wide threshold 02b used. Direction is
reported separately from significance: agreeing in sign without reaching the threshold is
a weaker but distinct statement, and worth being able to see.

In [ ]:
rep_cols <- c('lane_id','label','include','rsid','gene','outcome','exposure','metabolite',
              'MESA_beta','MESA_p','FHS_beta','FHS_se','FHS_p','n','same_direction','replicated')
if (!nrow(hits)) {
  rep_tbl <- setNames(data.table(matrix(nrow=0, ncol=length(rep_cols))), rep_cols)
  cat('No hit was testable in FHS, so the replication table is empty.\n')
} else {
p_replication <- 0.05 / nrow(hits)

rep_tbl <- hits[
  ][, same_direction := sign(fhs_est) == sign(mesa_beta)
  ][, replicated     := same_direction & fhs_p < p_replication
  ][order(fhs_p)
  ][, .(lane_id, label, include, rsid, gene, outcome=Y, exposure=E, metabolite=met_label,
        MESA_beta=signif(mesa_beta,3), MESA_p=signif(mesa_p,3),
        FHS_beta=signif(fhs_est,3), FHS_se=signif(fhs_se,3), FHS_p=signif(fhs_p,3),
        n=fhs_n, same_direction, replicated)
]

cat('Replication threshold: p <', signif(p_replication,3),
    ' (Bonferroni over', nrow(hits), 'tested hits)\n')
cat('Same direction as MESA:', rep_tbl[, sum(same_direction)], 'of', nrow(rep_tbl),
    '| replicated:', rep_tbl[, sum(replicated)], '\n\n')

}
if (nrow(rep_tbl))
  rep_tbl |> gt() |> tab_header('FHS replication of the MESA GxM hits') |> gt:::as.tags.gt_tbl()

# Metabolome-wide screen in FHS

The section above asks whether FHS reproduces the individual MESA hits. This one asks a
different question, and is the validity backstop for a lane rather than for a feature: run the
same screen FHS can support -- every MESA-tested metabolite that exists in the aligned panel --
and compare the two cohorts across the whole panel, not only at the top.

Two things it can show that a handful of per-hit tests cannot. Concordance: if the MESA screen
were noise, its effect estimates would be unrelated to FHS's, so an excess of same-sign estimates
among MESA's strongest is evidence the screen is measuring something. Coverage: how much of the
MESA panel FHS can even address, which bounds every replication claim in the paper. FHS carries
roughly a third of MESA's observations, so a feature-level null here is weak evidence of absence;
state it as such.

`SCREEN_TOP_K` sets the scope, because the cost is entirely in the number of fits and most of the
panel is noise in both cohorts. The default takes MESA's strongest features per lane -- enough for
sign concordance and for testing the features that matter -- at a few hundred fits. Set it to
`NULL` for the full alignable panel, which is what a metabolome-wide **meta-analysis** needs: FHS
adds about a third of MESA's sample, shrinking the standard error by roughly 13% and raising z by
about 15%. That is worth having for one lane and rarely worth it for four.

In [ ]:
# --- Scope ---------------------------------------------------------------------------------
SCREEN_TOP_K  <- 100    # MESA's strongest features per lane; NULL = the whole alignable panel.
SCREEN_LANES  <- NULL   # lane_ids to screen; NULL = every lane FHS can fit.
SCREEN_ADJUST_GE <- TRUE  # Add G*E where FHS has the exposure, so the estimate matches MESA's
                          # exactly and the two can be meta-analysed. The per-hit replication
                          # above deliberately omits it, to keep every lane comparable there.
SCREEN_KINSHIP <- FALSE   # FALSE: rank with lm and no relatedness term -- far faster, and which
                          # features rise to the top barely depends on it. Everything REPORTED is
                          # refit with kinship below, which is what FHS families require.
REFIT_TOP_N    <- 20      # Per lane, refit with kinship after a fast ranking pass.
CHUNK          <- 50      # Checkpoint every CHUNK fits, so a long run is never lost.

grab_file_if_not_extant('results/GxM_results.csv')  # The MESA screen, from 02b.
mesa_gxm <- ensure_lane_id(fread('results/GxM_results.csv'), lanes)[!is.na(lane_id) & !is.na(est)]
mesa_gxm[, M_aligned := sub('_(pos|neg)$', '', M)] # MESA ids carry an ionization suffix; see above.

# Screen only features MESA tested AND the aligned panel carries, so every FHS estimate has a
# MESA counterpart to sit beside. Amide-neg ids are positional and do not correspond across
# datasets, so they drop out here rather than being matched wrongly.
screen <- lanes[, .(lane_id, label, include, extra_covars)][
  mesa_gxm[M_aligned %in% met_nms & !grepl('_Amide_neg$', M),
           .(lane_id, Y, E, G, M, M_aligned, mesa_beta=est, mesa_se=se, mesa_p=p)], on='lane_id']
screen <- screen[G %in% anchor_snps & Y %in% names(data)]
if (!is.null(SCREEN_LANES)) screen <- screen[lane_id %in% SCREEN_LANES]
if (!is.null(SCREEN_TOP_K)) screen <- screen[order(mesa_p), head(.SD, SCREEN_TOP_K), by=lane_id]

cat('Panel coverage, MESA -> FHS:\n')
print(mesa_gxm[, .(mesa_tested = .N,
                   in_aligned  = sum(M_aligned %in% met_nms & !grepl('_Amide_neg$', M)),
                   screened    = sum(M %in% screen$M)), by=.(lane_id, Y, E)])
stopifnot('no lane is screenable in FHS' = nrow(screen) > 0)

# Same specification as the replication fits above, including the omitted G:E adjustment, so a
# screened estimate and a replicated one are the same quantity.
screen[, fmla := paste0(Y, ' ~ ', G, '*', M_aligned)
  ][, fmla := paste(fmla, '+', mapply(Y, E, fifelse(is.na(extra_covars), '', extra_covars), USE.NAMES=FALSE,
        FUN=\(Y, E, x) paste(lane_covars(covars, Y, E, x), collapse='+')))
  ][, fmla := paste(fmla, '+', paste0(G, ':sex'))
  ][, fmla := paste(fmla, '+', paste(collapse='+', paste0('gPC', 1:9, '*', M_aligned)))
  ][, fmla := fifelse(SCREEN_ADJUST_GE & E %in% names(data), paste0(fmla, ' + ', G, '*', E), fmla)
  ][, fmla := paste(fmla, '+ (1|NWD_Id)')
  ][, term_pat := paste0(G, ':', M_aligned, '|', M_aligned, ':', G)
]
cat('\nScope: ', if (is.null(SCREEN_TOP_K)) 'the whole alignable panel' else
      paste('MESA\'s top', SCREEN_TOP_K, 'features per lane'),
    '; G:E adjustment ', if (SCREEN_ADJUST_GE) 'on where FHS has the exposure' else 'off', '.\n', sep='')
cat('Fits queued: ', nrow(screen), '. The next cell times one fit and extrapolates.\n', sep='')

In [ ]:
# One fork per fit (mc.preschedule=FALSE) and errors captured as text, for the reason recorded
# in 02b: with prescheduling, one dead worker takes its whole batch of results with it.
# One fit, with or without the kinship term. lm() reports no denominator df, so the residual
# df is inserted to keep the same six-value shape.
fit_fhs <- \(fmla, pat, kinship = SCREEN_KINSHIP) tryCatch({
  if (kinship) {
    model <- myLmer(fmla, data, list(NWD_Id=km))
    coefs <- summary(model)$coefficients; n <- nrow(model@frame)
  } else {
    model <- lm(sub(' + (1|NWD_Id)', '', fmla, fixed=TRUE), data)
    co    <- summary(model)$coefficients
    coefs <- cbind(co[, 1:2, drop=FALSE], df = model$df.residual, co[, 3:4, drop=FALSE])
    n     <- nrow(model$model)
  }
  coefs <- coefs[grepl(pat, rownames(coefs)), , drop=FALSE]
  if (nrow(coefs) != 1) stop('interaction term not estimated')
  c(coefs[1,], n)
}, error = \(e) conditionMessage(e))

collect <- \(fits) {
  ok  <- vapply(fits, \(r) is.numeric(r) && length(r) == 6, TRUE)
  err <- rep(NA_character_, length(fits))
  err[!ok] <- vapply(fits[!ok], \(r) if (length(r)) paste(as.character(r), collapse=' ') else 'no result (worker died?)', '')
  fits[!ok] <- list(rep(NA_real_, 6))
  c(setNames(transpose(fits), c('fhs_est','fhs_se','fhs_df','fhs_t','fhs_p','fhs_n')), list(fit_error=err))
}

res_cols <- c('fhs_est','fhs_se','fhs_df','fhs_t','fhs_p','fhs_n','fit_error')
cores <- min(4L, parallel::detectCores())
options(mc.cores = cores)

# Timed, not guessed: the cost per fit depends on the model, on the kinship factorization and
# on how many threads the BLAS takes -- assuming it was wrong by 50x on 2026-09-20.
one <- system.time(fit_one <- fit_fhs(screen$fmla[1], screen$term_pat[1]))[['elapsed']]
cat('One fit: ', round(one, 1), ' s (kinship = ', SCREEN_KINSHIP, '). ', nrow(screen),
    ' fits over ', cores, ' cores is roughly ', round(nrow(screen) * one / 60 / cores),
    ' minutes.\n', sep='')
if (!is.numeric(fit_one)) cat('NB that fit failed:', fit_one, '\n')

# Resume from the checkpoint: rows already fitted are not fitted again, so an interrupted
# run continues rather than starting over.
screen[, (res_cols) := .(NA_real_, NA_real_, NA_real_, NA_real_, NA_real_, NA_real_, NA_character_)]
if (file.exists('results/fhs_gxm_screen.csv')) {
  done <- fread('results/fhs_gxm_screen.csv')
  if (all(c('lane_id','M',res_cols) %in% names(done)))
    screen[done[!is.na(fhs_est), c('lane_id','M',res_cols), with=FALSE],
           on=.(lane_id, M), (res_cols) := mget(paste0('i.', res_cols))]
}
todo <- screen[, which(is.na(fhs_est) & is.na(fit_error))]
cat(nrow(screen) - length(todo), 'fits already in the checkpoint;', length(todo), 'to run\n')

t0 <- Sys.time()
for (ch in split(todo, ceiling(seq_along(todo)/CHUNK))) {
  screen[ch, (res_cols) := collect(mcMap(fmla, term_pat, f=fit_fhs, mc.preschedule=FALSE))]
  fwrite(screen, 'results/fhs_gxm_screen.csv')   # Checkpoint every chunk.
  cat('  ', sum(!is.na(screen$fhs_est)), '/', nrow(screen), ' fits, ',
      round(difftime(Sys.time(), t0, units='mins')), ' min elapsed\r', sep='')
}
cat('\n', screen[, sum(!is.na(fit_error))], 'of', nrow(screen), 'fits failed\n')

# Everything reported is refit with kinship: a ranking pass without it is a ranking pass only.
if (!SCREEN_KINSHIP && REFIT_TOP_N > 0) {
  top <- screen[!is.na(fhs_p)][order(fhs_p), head(.SD, REFIT_TOP_N), by=lane_id][, .(lane_id, M)]
  if ('model' %in% names(screen))   # A resumed run: do not refit what is already kinship-fitted.
    top <- top[!screen[model %like% 'kinship', .(lane_id, M)], on=.(lane_id, M)]
  cat('Refitting', nrow(top), 'top features with kinship...\n')
  screen[top, on=.(lane_id, M), (res_cols) :=
           collect(mcMap(fmla, term_pat, f=\(f, p) fit_fhs(f, p, kinship=TRUE), mc.preschedule=FALSE))]
  screen[, model := fifelse(paste(lane_id, M) %in% top[, paste(lane_id, M)],
                            'kinship (relmatLmer)', 'ranking only (lm, no relatedness)')]
  fwrite(screen, 'results/fhs_gxm_screen.csv')
}
Sys.time() - t0

In [ ]:
screen <- fread('results/fhs_gxm_screen.csv')[!is.na(fhs_est)] # Read checkpoint.

# FHS's own screen threshold, by the estimator 02b uses (Li & Ji 2005) over the features
# actually screened here -- not MESA's, which was computed on a larger panel. With
# SCREEN_TOP_K set these are pre-selected features, so this is a yardstick, not a screen
# threshold: the honest correction for a top-K subset is Bonferroni over the K tested.
met_scr <- unique(screen$M_aligned)
met_mtx <- as.matrix(data[rowSums(is.na(data[, ..met_scr])) == 0, ..met_scr])
lam_fhs <- prcomp(met_mtx, scale=TRUE, center=TRUE)$sdev^2
m_eff_fhs <- sum((lam_fhs >= 1) + (lam_fhs - floor(lam_fhs)))
p_screen_fhs <- 0.05 / m_eff_fhs
cat('FHS screen: ', length(met_scr), ' features, M_eff (Li & Ji) = ', round(m_eff_fhs),
    ', threshold p < ', signif(p_screen_fhs, 3), '\n', sep='')

# Concordance with MESA. Under a null screen the two cohorts' estimates are unrelated, so a
# correlation across features, or an excess of same-sign estimates among MESA's strongest,
# is evidence the screen carries signal. TOP_K is fixed here rather than tuned.
TOP_K <- 20
conc <- screen[, {
  top <- .SD[order(mesa_p)][1:min(TOP_K, .N)]
  same <- top[, sum(sign(fhs_est) == sign(mesa_beta))]
  .(features   = .N,
    hits_fhs   = sum(fhs_p < p_screen_fhs),
    hits_bonf  = sum(fhs_p < 0.05 / .N),   # Bonferroni over what was actually tested here.
    r_beta     = round(cor(mesa_beta, fhs_est), 3),
    r_z        = round(cor(mesa_beta/mesa_se, fhs_est/fhs_se), 3),
    top_k      = nrow(top),
    same_sign  = same,
    sign_p     = signif(binom.test(same, nrow(top), 0.5)$p.value, 3),
    top_fhs_p  = signif(top[1, fhs_p], 3))
}, by=.(lane_id, label, Y, E)]
print(conc)

# Inverse-variance meta-analysis of the two cohorts, for the features screened here. Valid
# only where the FHS model matches MESA's, i.e. with SCREEN_ADJUST_GE on and the exposure
# present; flagged per row rather than assumed.
screen[, comparable := SCREEN_ADJUST_GE & E %in% names(data)
  ][, w_mesa := 1/mesa_se^2][, w_fhs := 1/fhs_se^2
  ][, meta_beta := (mesa_beta*w_mesa + fhs_est*w_fhs) / (w_mesa + w_fhs)
  ][, meta_se   := sqrt(1/(w_mesa + w_fhs))
  ][, meta_p    := 2*pnorm(-abs(meta_beta/meta_se))
]
cat('\nMeta-analysis (MESA + FHS), smallest p per lane:\n')
print(screen[comparable == TRUE][order(meta_p), head(.SD, 3), by=lane_id
  ][, .(lane_id, metabolite=M, MESA_p=signif(mesa_p,3), FHS_p=signif(fhs_p,3),
        meta_beta=signif(meta_beta,3), meta_p=signif(meta_p,3))])

# MESA's strongest features in each lane, beside their FHS estimates.
screen[order(mesa_p), head(.SD, 5), by=lane_id
  ][, .(lane_id, metabolite=M, MESA_beta=signif(mesa_beta,3), MESA_p=signif(mesa_p,3),
        FHS_beta=signif(fhs_est,3), FHS_se=signif(fhs_se,3), FHS_p=signif(fhs_p,3),
        same_direction=sign(fhs_est)==sign(mesa_beta), n=fhs_n)
] |> gt() |> tab_header('MESA top features, screened in FHS') |> gt:::as.tags.gt_tbl()

## Write

In [ ]:
fwrite(rep_tbl,   'results/fhs_replication.csv')       # Tested hits, MESA beside FHS.
fwrite(mesa_hits, 'results/fhs_replication_full.csv')  # All hits, including why any was skipped.
fwrite(conc,      'results/fhs_screen_concordance.csv') # Per-lane coverage and MESA-FHS concordance.

system(paste0('gcloud storage cp results/fhs_replication*.csv ', ws_bucket, '/results/'))
system(paste0('gcloud storage cp results/fhs_gxm_screen.csv results/fhs_screen_concordance.csv ', ws_bucket, '/results/'))